In [ ]:
# -*- coding: utf-8 -*-
import os
import sys
os.environ["PYSPARK_SUBMIT_ARGS"]='--conf spark.sql.catalogImplementation=in-memory pyspark-shell'
os.environ["PYSPARK_PYTHON"]='/opt/anaconda/envs/bd9/bin/python'
os.environ["SPARK_HOME"]='/usr/hdp/current/spark2-client'

spark_home = os.environ.get('SPARK_HOME', None)
if not spark_home:
    raise ValueError('SPARK_HOME environment variable is not set')
sys.path.insert(0, os.path.join(spark_home, 'python'))
sys.path.insert(0, os.path.join(spark_home, 'python/lib/py4j-0.10.7-src.zip'))
exec(open(os.path.join(spark_home, 'python/pyspark/shell.py')).read())


from pyspark.sql.types import *

In [ ]:
spark.stop()

In [ ]:
from pyspark import SparkConf, SparkContext, SQLContext
from pyspark import sql
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import *
import pyspark.sql.functions as F
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.feature import Word2Vec
import pandas as pd

    Load the whole datasets

In [ ]:
train_path = os.path.join('/labs/lab10data/', 'lab10_train.csv')
item_path = os.path.join('/labs/lab10data/', 'lab10_items.csv')
test_path = os.path.join('/labs/lab10data/', 'lab10_test.csv')
views_programmes = os.path.join('/labs/lab10data', 'lab10_views_programmes.csv')
res_path = os.path.join('/user/artem.moskalets/')

In [ ]:
train_fields = [StructField('user_id', LongType(), True),
                StructField('item_id', LongType(), True),
               StructField('purchase', ByteType(), True)]
schema_train = StructType(train_fields)
df_train = spark.read.csv(header = 'true', schema = schema_train, path = train_path, sep=',')

In [ ]:
df_train.printSchema()

In [ ]:
df_train.show()

In [ ]:
item_fields = [StructField('item_id', LongType(), True),
               StructField('channel_id', DoubleType(), True),
               StructField('datetime_availability_start', TimestampType(), True),
               StructField('datetime_availability_stop', TimestampType(), True),
               StructField('datetime_show_start', TimestampType(), True),
               StructField('datetime_show_stop', TimestampType(), True),
               StructField('content_type', ByteType(), True),
               StructField('title', StringType(), True),
               StructField('year', DoubleType(), True),
               StructField('genres', StringType(), True),
               StructField('region_id', FloatType(), True)]               
schema_item = StructType(item_fields)
df_item = spark.read.csv(header = 'true', schema = schema_item, path = item_path, sep='\t')

In [ ]:
names = df_item. select('genres')
names.show()

In [ ]:
names = names.withColumn('genres_s',F.split("genres",','))
names.printSchema()

In [ ]:
names = names.select(F.explode('genres_s').alias('genres_s')).groupby().agg(F.collect_set('genres_s'))
names_genres = names.rdd.flatMap(lambda x: x).collect()

In [ ]:
for i in names_genres:
    for j in sorted(i):
        print (j)

In [ ]:
dictionary = {u'???-????': u'???????',             
u'???????': u'???????',\
u'???????': u'??????',\
u'???????': u'???????',\
u'?????????': u'?????????, ????',\
u'?????????':u'?????????',\
u'??????????????': u'??????????????',\
u'?????': u'?????',\
u'????????????': u'????????????',\
u'???????': u'???????',\
u'????????????????':u'???????????????',
u'?????????': u'?????????',\
u'???????????': u'???????????',\
u'???????????': u'???????????',\
u'????????': u'????????,???????????',\
u'???????????': u'???????????',\
u'?????????????': u'?????????',\
u'????????': u'????????',\
u'??? ???? ?????': u'????????',\
u'????????? ????': u'?????????, ???????',\
u'????': u'???????',\
u'??????????': u'?????',\
u'?????': u'???????, ?????',\
u'??????????': u'??????????????',\
u'?????? ? 3d': u'??????,3d',\
u'???????': u'???????,??????????????',\
u'??????? ??????????': u'???????,??????????????',\
u'???? ??? ?????': u'??????????????, ???????????',\
u'??????????????': u'??????????????, ???????????',\
u'??????? ?????': u'???????,?????',\
u'??? ?????': u'???????',\
u'??? ????? ?????????': u'???????',\
u'???????? ???????????': u'???????, ??????????,????????',\
u'???????? ???????' : u'????????, ???????',
u'??????? ???????' : u'???????, ???????',             
u'??????????': u'??????????,????????',
u'????????????': u'???????, ??????????',\
u'???????????': u'???????,??????????',\
u'?????????? ? 3d': u'???????,??????????, 3d',\
u'??????? ???????????': u'???????,??????????,???????',\
u'???????????': u'???????,???????????',\
u'??????': u'???????,??????',\
u'??????': u'???????,??????',\
u'????????': u'???????',\
u'??????????????': u'???????,??????????',\
u'????????': u'???????,??????????',\
u'???????': u'???????, ??? ????????'}


In [ ]:
test_fields = [StructField('user_id', LongType(), True),
              StructField('item_id', LongType(), True),
              StructField('purchase', ByteType(), True)]
schema_test = StructType(test_fields)
df_test = spark.read.csv(header = 'true', schema = schema_test, path = test_path, sep=',')

In [ ]:
df_test.printSchema()

In [ ]:
views_programmes_fields = [StructField('user_id', LongType(), True),
                           StructField('item_id', LongType(), True),
                           StructField('ts_start', LongType(), True),
                           StructField('ts_end', LongType(), True),
                           StructField('item_type', StringType(), True)]
schema_views_programmes = StructType(views_programmes_fields)
df_views_programmes = spark.read.csv(header = 'true', schema = schema_views_programmes, path = views_programmes, sep=',')

In [ ]:
df_views_programmes.printSchema()

    Feature engineering

In [ ]:
df_features = df_train.join(df_item, df_train.item_id == df_item.item_id, "inner").drop(df_item.item_id)

In [ ]:
df_features = df_features.select('user_id','item_id','purchase','year','genres')

In [ ]:
df_features = df_features.select('*', coalesce(df_features["year"], lit(0.0)).alias('year_0'))


In [ ]:
df_features = df_features.drop("year").select('*', col('year_0').alias("year")).drop('year_0')

In [ ]:
df_features = df_features.fillna('noname')

In [ ]:
def gen(films):
    lst =  films.split(',')
    res = ''
    for l in lst:
        l = l.lower().strip()
        for key in list(dictionary.keys()):
            l = l.replace(key, dictionary[key])
        res = res + ','+ l +','
    return res
gen =  udf(gen,StringType())  

In [ ]:
df_features = df_features.withColumn('Words', gen(df_features.genres))

In [ ]:
df_features.select(df_features.Words).show()

In [ ]:
def un(one, two, three, four):
    return str(one)+','+str(two) +','+str(three)+','+(four)
un = udf(un,StringType())
df_features = df_features.withColumn('all_features', un('user_id','item_id','year','words'))

In [ ]:
df_features.select('all_features').show(truncate = False)

In [ ]:
df_features.select(df_features.Words).count()

In [ ]:
from pyspark.ml.feature import HashingTF, IDF, Tokenizer, RegexTokenizer

tokenizer = RegexTokenizer(inputCol="all_features", 
                           outputCol="lst", 
                           gaps=False, 
                           pattern=r'[a-zA-Z?-??-??0-9_\-]{2,}')
df_features = tokenizer.transform(df_features)

In [ ]:
df_features.select('lst').show(truncate = False)

In [ ]:
from pyspark.ml.feature import CountVectorizer
from pyspark.ml.linalg import Vectors

In [ ]:
cv = CountVectorizer(inputCol= "lst", outputCol="features")
model_cv = cv.fit(df_features)

In [ ]:
df_features = model_cv.transform(df_features)

In [ ]:
df_features.show()

In [ ]:
df_features = df_features.drop('genres','Words','user_id','item_id','all_features','genres','year','lst')

In [ ]:
df_features.show()

In [ ]:
# Index labels, adding metadata to the label column.
from pyspark.ml.feature import StringIndexer, VectorIndexer

# Fit on whole dataset to include all labels in index.
labelIndexer = StringIndexer(inputCol="purchase", outputCol="indexedLabel").fit(df_features)
# Automatically identify categorical features, and index them.
# Set maxCategories so features with > 4 distinct values are treated as continuous.
featureIndexer =\
VectorIndexer(inputCol="features", outputCol="indexedFeatures").fit(df_features)

# Split the data into training and test sets (30% held out for testing)
(trainingData, testData) = df_features.randomSplit([0.7, 0.3])

In [ ]:
from pyspark.ml.classification import RandomForestClassifier, RandomForestClassificationModel

# Train a RandomForest model.
rf = RandomForestClassifier(numTrees = 800, maxDepth = 12, labelCol="purchase", featuresCol="features")

In [ ]:
# Chain indexers and forest in a Pipeline
pipeline = Pipeline(stages=[labelIndexer, featureIndexer, rf])

In [ ]:
# Train model.  This also runs the indexers.
model = rf.fit(df_features)

In [ ]:
model.save('RF_lab10_3.sav')

In [ ]:
model = RandomForestClassificationModel.load('RF_lab10_3.sav')

In [ ]:
df_test.show()

In [ ]:
df_test = spark.read.csv(header = 'true', schema = schema_test, path = test_path, sep=',')
df_test = df_test.join(df_item, df_test.item_id == df_item.item_id, "inner").drop(df_item.item_id)
df_test = df_test.select('user_id','item_id','year','genres')
df_test = df_test.select('*', coalesce(df_test["year"], lit(0.0)).alias('year_0'))
df_test = df_test.drop("year").select('*', col('year_0').alias("year")).drop('year_0')
df_test = df_test.fillna('noname')
df_test = df_test.withColumn('Words', gen(df_test.genres))
df_test = df_test.withColumn('all_features', un('user_id','item_id','year','words'))
df_test = tokenizer.transform(df_test)
df_test = model_cv.transform(df_test)
df_test = df_test.drop('genres','Words','all_features','genres','year','lst','purchase')

In [ ]:
df_test.show()

In [ ]:
df_test.count()

In [ ]:
2156840

In [ ]:
df_test.show()

In [ ]:
predictions = model.transform(df_test)

In [ ]:
def proba1(vector):
    from pyspark.ml.linalg import Vectors
    res = float(vector[1])
    return res
proba1 = udf(proba1, FloatType())

In [ ]:
predictions1 = predictions.select('*', proba1('probability').alias('purchase'))

In [ ]:
predictions1.show()

In [ ]:
predictions1.count()

In [ ]:
Res = predictions1.select("user_id","item_id","purchase").\
                    sort("user_id","item_id")

In [ ]:
Res.coalesce(1).write.mode("overwrite").option("header", "true").csv("lab10_3.csv")

In [ ]:
!hadoop fs -getmerge lab10_3.csv lab10s.csv